### Here We processed as we want , but at the end we were only 1812 datapoints (users) who have given more than 2 reviews so we took electronics data also and combinin both the things

In [ ]:
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
import os
import json
from tqdm.auto import tqdm

In [ ]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
input_file = "../Data/raw/meta.jsonl"
output_file = "../Data/processed/meta_cleaned.jsonl"

remove_fields = {
    "images",
    "videos",
    "store",
    "categories",
    "bought_together"
}

with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:

    for line in tqdm(infile, desc="Cleaning product metadata"):
        if not line.strip():
            continue

        product = json.loads(line)

        for field in remove_fields:
            product.pop(field, None)

        outfile.write(json.dumps(product, ensure_ascii=False) + "\n")

In [ ]:
with open(output_file, "r", encoding="utf-8") as f:
    first_product = json.loads(next(f))

print(first_product)
print("\nRemaining fields:")
print(first_product.keys())

In [ ]:

input_file = "../Data/processed/meta_cleaned.jsonl"
output_file = "../Data/processed/meta_filtered.jsonl"

# Count total products so tqdm can show proper progress
with open(input_file, "r", encoding="utf-8") as f:
    total_products = sum(1 for line in f if line.strip())

# Counters
removed_zero_reviews = 0
removed_null_price = 0
removed_missing_asin = 0
total_removed = 0

with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:

    for line in tqdm(
        infile,
        total=total_products,
        desc="Filtering products",
        colour="green"
    ):
        if not line.strip():
            continue

        product = json.loads(line)

        parent_asin = product.get("parent_asin")
        # price = product.get("price")
        review_count = product.get("rating_number")

        # Check every removal condition
        missing_asin = (
            parent_asin is None or
            str(parent_asin).strip() == ""
        )

        # null_price = (
        #     price is None or
        #     str(price).strip() == ""
        # )

        zero_reviews = (
            review_count is not None and
            review_count <= 0
        )

        # Count each reason independently
        if zero_reviews:
            removed_zero_reviews += 1

        # if null_price:
        #     removed_null_price += 1

        if missing_asin:
            removed_missing_asin += 1

        # Remove product if any condition is true
        if zero_reviews  or missing_asin:
            total_removed += 1
            continue

        # Keep valid product
        outfile.write(
            json.dumps(product, ensure_ascii=False) + "\n"
        )

print("\nFiltering completed!")
print(f"Products removed because of 0 reviews : {removed_zero_reviews}")
print(f"Products removed because of null price : {removed_null_price}")
print(f"Products removed because of missing ASIN: {removed_missing_asin}")
print(f"Total unique products removed         : {total_removed}")
print(f"Products remaining                    : {total_products - total_removed}")

In [ ]:
meta_file = "../Data/processed/meta_filtered.jsonl"
reviews_file = "../Data/raw/reviews.jsonl"
output_file = "../Data/processed/reviews_filtered.jsonl"

valid_asins = set()

with open(meta_file, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading valid product ASINs", colour="green"):
        if not line.strip():
            continue

        product = json.loads(line)

        parent_asin = product.get("parent_asin")

        if parent_asin:
            valid_asins.add(parent_asin)

print(f"Valid product ASINs: {len(valid_asins):,}")

In [ ]:
# First count total reviews for a proper progress bar
with open(reviews_file, "r", encoding="utf-8") as f:
    total_reviews = sum(1 for line in f if line.strip())

kept_reviews = 0
removed_reviews = 0
invalid_json = 0

with open(reviews_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:

    for line in tqdm(
        infile,
        total=total_reviews,
        desc="Filtering reviews",
        colour="green"
    ):
        if not line.strip():
            continue

        try:
            review = json.loads(line)
        except json.JSONDecodeError:
            invalid_json += 1
            continue

        parent_asin = review.get("parent_asin")

        if parent_asin in valid_asins:
            outfile.write(
                json.dumps(review, ensure_ascii=False) + "\n"
            )
            kept_reviews += 1
        else:
            removed_reviews += 1

print("\nFiltering completed!")
print(f"Total reviews    : {total_reviews:,}")
print(f"Reviews kept     : {kept_reviews:,}")
print(f"Reviews removed  : {removed_reviews:,}")
print(f"Invalid JSON     : {invalid_json:,}")

In [ ]:
input_file = "../Data/processed/reviews_filtered.jsonl"
output_file = "../Data/processed/reviews_cleaned.jsonl"

remove_fields = {
    "asin",
    "images",
    "helpful_vote",
    "verified_purchase"
}

# Count reviews for progress bar
with open(input_file, "r", encoding="utf-8") as f:
    total_reviews = sum(1 for line in f if line.strip())

with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:

    for line in tqdm(
        infile,
        total=total_reviews,
        desc="Cleaning reviews",
        colour="green"
    ):
        if not line.strip():
            continue

        review = json.loads(line)

        for field in remove_fields:
            review.pop(field, None)

        outfile.write(
            json.dumps(review, ensure_ascii=False) + "\n"
        )

print("\nReview cleaning completed!")
print(f"Total reviews processed: {total_reviews:,}")

In [ ]:
with open(output_file, "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline()

        if not line:
            break

        review = json.loads(line)

        print(f"\n{'=' * 80}")
        print(f"Review {i + 1}")
        print(f"{'=' * 80}")
        print(json.dumps(review, indent=2, ensure_ascii=False))

In [ ]:
from collections import Counter
reviews_file = "../Data/processed/reviews_cleaned.jsonl"

user_review_count = Counter()

# Count total reviews for progress bar
with open(reviews_file, "r", encoding="utf-8") as f:
    total_reviews = sum(1 for line in f if line.strip())

# Count reviews per user
with open(reviews_file, "r", encoding="utf-8") as f:
    for line in tqdm(
        f,
        total=total_reviews,
        desc="Counting user reviews",
        colour="green"
    ):
        if not line.strip():
            continue

        review = json.loads(line)

        user_id = review.get("user_id")

        if user_id:
            user_review_count[user_id] += 1

# Users having more than 2 reviews
users_more_than_2 = sum(
    1 for count in user_review_count.values()
    if count > 3
)

print(f"Total unique users       : {len(user_review_count):,}")
print(f"Users with > 3 reviews   : {users_more_than_2:,}")

In [ ]:
from collections import defaultdict, Counter
import json
from tqdm.notebook import tqdm

reviews_file = "../Data/processed/reviews_cleaned.jsonl"

user_reviews = defaultdict(set)

with open(reviews_file, "r", encoding="utf-8") as f:
    total_reviews = sum(1 for line in f if line.strip())

with open(reviews_file, "r", encoding="utf-8") as f:
    for line in f :
        if not line.strip():
            continue

        review = json.loads(line)

        user_id = review.get("user_id")
        parent_asin = review.get("parent_asin")

        if user_id and parent_asin:
            user_reviews[user_id].add(parent_asin)

thresholds = [1, 2, 3, 4, 5,6,7,8,9, 10]

print("\nUsers with at least N DISTINCT products:\n")

for n in thresholds:
    count = sum(
        1 for products in user_reviews.values()
        if len(products) >= n
    )
    print(f">= {n} products : {count:,}")